In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# Load the IMDB dataset
max_features = 20000  # Number of words to consider as features
maxlen = 100  # Cut texts after this number of words (for padding)
batch_size = 32

print('Loading data...')
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(f'{len(x_train)} train sequences')
print(f'{len(x_test)} test sequences')

print('Pad sequences (samples x time)')
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)
print(f'x_train shape: {x_train.shape}')
print(f'x_test shape: {x_test.shape}')

Loading data...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
25000 train sequences
25000 test sequences
Pad sequences (samples x time)
x_train shape: (25000, 100)
x_test shape: (25000, 100)


In [3]:
model = Sequential()
model.add(Embedding(max_features, 128, input_length=maxlen))
model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [4]:
print('Training model...')
history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=15,
                    validation_data=(x_test, y_test))

Training model...
Epoch 1/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 18ms/step - accuracy: 0.7381 - loss: 0.5085 - val_accuracy: 0.8156 - val_loss: 0.3987
Epoch 2/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8987 - loss: 0.2532 - val_accuracy: 0.8440 - val_loss: 0.3645
Epoch 3/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.9421 - loss: 0.1629 - val_accuracy: 0.8427 - val_loss: 0.3891
Epoch 4/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.9640 - loss: 0.1023 - val_accuracy: 0.8316 - val_loss: 0.5323
Epoch 5/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9763 - loss: 0.0716 - val_accuracy: 0.8320 - val_loss: 0.5329
Epoch 6/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.9856 - loss: 0.0430 - val_accuracy: 0.8268 - val_loss: 0.6367
Epoch 7/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.9898 - loss: 0.0335 - val_accuracy: 0.8272 - val_loss: 0.7589
Epoch 8/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9928

In [5]:
score, acc = model.evaluate(x_test, y_test,
                            batch_size=batch_size)
print(f'Test score: {score}')
print(f'Test accuracy: {acc}')

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8197 - loss: 1.0512
Test score: 1.0422385931015015
Test accuracy: 0.8225200176239014


In [6]:
# Decode and display review text along with its predicted sentiment
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
# Select a sample review for prediction
sample_review = x_test[2000]
decoded_review = decode_review(sample_review)

In [8]:
# Predict sentiment using the trained LSTM model
prediction = model.predict(np.expand_dims(sample_review, axis=0))
sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"

print("\nSample Review:")
print(f"Review: {decoded_review}")
print(f"Predicted Sentiment: {sentiment}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step

Sample Review:
Review: this one brilliantly you can't help but wonder if he is really out there i reckon he and the other main cast members probably had nightmares for weeks after doing this movie as it's so intense when i first saw it i was just ? channels on the remote late one evening i got hooked within minutes look up www answers com for ? ? who is the character that carlos the is based on for both i remember reading about ? arrest in the paper in 1997 it was front page for weeks through the trial after his arrest
Predicted Sentiment: Positive


In [9]:
# Select a sample review for prediction
sample_review = x_test[100]
decoded_review = decode_review(sample_review)

In [10]:
# Predict sentiment using the trained LSTM model
prediction = model.predict(np.expand_dims(sample_review, axis=0))
sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"

print("\nSample Review:")
print(f"Review: {decoded_review}")
print(f"Predicted Sentiment: {sentiment}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

Sample Review:
Review: you actually want to die however delivers considerably more br br rather than focus on bare flesh and gore though there is a little of each no sex however the flick focuses on delivering impending dread mounting tension amidst a lovely scenic backdrop these feelings are further heightened by a cast of realistically likable characters and antagonists that are more amoral than cardboard ? of evil oh yeah george kennedy is here too and when is that not a good thing br br if you liked wrong turn then watch this to see where much of its' ? came from
Predicted Sentiment: Negative
